# 1. VQE Comparison - Estimators

Here we will compare between the following estimators: 

1. Aer (Statevector) - Mathematically ideal, exact estimator (Infinite shots, no noise).
2. Aer (Sampling Estimator) -  Finite number of shots and introduction of shot noise and seeds. 

‎ 

3. Estimator V1 (Statevector) - Ideal simulation. 
4. Estimator V1 (Sampling) - Finite number of shots and introduction of shot noise and seeds. 

‎ 

5. BackendEstimator (Simulated) - BackendEstimator class connected to local simulator.

In [ ]:
from requirements_deprecated import *

In [ ]:
def VQE_comparison_estimators(dist, method="ref_exact", shots=None, seed=42):
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
        
    if method == "ref_exact":
    
        estimator = EstimatorV1()
        
    elif method == "ref_sampling":
            
        estimator = EstimatorV1(options={"shots": shots, "seed": seed})
        
    elif method == "aer_exact":
        
        estimator = AerEstimator(
            run_options={"shots": None}, 
            approximation=True
        )

    elif method == "aer_sampling":
                
        estimator = AerEstimator(
            run_options={"shots": shots, "seed": seed}, 
            approximation=False
        )

    elif method == "backend_wrapper":
                
        sim_backend = AerSimulator()
        estimator = BackendEstimatorV1(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)
    
    optimizer = SLSQP(maxiter=50)
    
    vqe = VQE(estimator, ansatz, optimizer)
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    return result.total_energies[0]

In [ ]:
distances = np.concatenate([np.linspace(0.4, 0.9, 5), np.linspace(1.0, 2.5, 5)])
distances = np.sort(distances)

results_comparison = {
    "EstimatorV1 (Exact)": [],
    "EstimatorV1 (Sampling)": [],
    "AerEstimator (Exact)": [],
    "AerEstimator (Sampling)": [],
    "BackendEstimatorV1 (Wrapper)": []
}

for d in tqdm(distances):
    
    try: results_comparison["EstimatorV1 (Exact)"].append(VQE_comparison_estimators(d, "ref_exact"))
    except: results_comparison["EstimatorV1 (Exact)"].append(None)

    try: results_comparison["EstimatorV1 (Sampling)"].append(VQE_comparison_estimators(d, "ref_sampling", shots=1024))
    except: results_comparison["EstimatorV1 (Sampling)"].append(None)
    
    try: results_comparison["AerEstimator (Exact)"].append(VQE_comparison_estimators(d, "aer_exact"))
    except: results_comparison["AerEstimator (Exact)"].append(None)

    try: results_comparison["AerEstimator (Sampling)"].append(VQE_comparison_estimators(d, "aer_sampling", shots=1024))
    except: results_comparison["AerEstimator (Sampling)"].append(None)

    try: results_comparison["BackendEstimatorV1 (Wrapper)"].append(VQE_comparison_estimators(d, "backend_wrapper", shots=1024))
    except: results_comparison["BackendEstimatorV1 (Wrapper)"].append(None)



In [ ]:
save_data(results_comparison, "H2_vqe_estimators_comparison_results")

‎ 

<div style="background-color: #505052ff; color: #ffffff; padding: 10px; width: 95%; border-radius: 5px;">

We got a a very inconsistent Backend Wrapper method as well as the sampling from Aer. So the main solve is to chnage the optimizers, using SPSA for sampling. 

</div>

‎ 

In [ ]:
def VQE_comparison_estimators_Optimizer(dist, method="ref_exact", shots=None, seed=42):
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    
    if method == "ref_exact":

    
        estimator = EstimatorV1()
        
    elif method == "ref_sampling":
    
        
        estimator = EstimatorV1(options={"shots": shots, "seed": seed})
        
    elif method == "aer_exact":

        
        estimator = AerEstimator(
            run_options={"shots": None}, 
            approximation=True
        )

    elif method == "aer_sampling":
        
        
        estimator = AerEstimator(
            run_options={"shots": shots, "seed": seed}, 
            approximation=False
        )

    elif method == "backend_wrapper":
        
        
        sim_backend = AerSimulator()
        estimator = BackendEstimatorV1(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)

    
    if "exact" in method:
    
        optimizer = SLSQP(maxiter=50)
    
    else:
    
        optimizer = COBYLA(maxiter=100)

    
    vqe = VQE(estimator, ansatz, optimizer)
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    return result.total_energies[0]

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

results_comparison_optimizer = {
    "EstimatorV1 (Exact)": [],
    "EstimatorV1 (Sampling)": [],
    "AerEstimator (Exact)": [],
    "AerEstimator (Sampling)": [],
    "BackendEstimatorV1 (Wrapper)": []
}

for d in tqdm(distances):
    
    try: results_comparison_optimizer["EstimatorV1 (Exact)"].append(VQE_comparison_estimators_Optimizer(d, "ref_exact"))
    except: results_comparison_optimizer["EstimatorV1 (Exact)"].append(None)

    try: results_comparison_optimizer["EstimatorV1 (Sampling)"].append(VQE_comparison_estimators_Optimizer(d, "ref_sampling", shots=1024))
    except: results_comparison_optimizer["EstimatorV1 (Sampling)"].append(None)
    
    try: results_comparison_optimizer["AerEstimator (Exact)"].append(VQE_comparison_estimators_Optimizer(d, "aer_exact"))
    except: results_comparison_optimizer["AerEstimator (Exact)"].append(None)

    try: results_comparison_optimizer["AerEstimator (Sampling)"].append(VQE_comparison_estimators_Optimizer(d, "aer_sampling", shots=1024))
    except: results_comparison_optimizer["AerEstimator (Sampling)"].append(None)

    try: results_comparison_optimizer["BackendEstimatorV1 (Wrapper)"].append(VQE_comparison_estimators_Optimizer(d, "backend_wrapper", shots=1024))
    except: results_comparison_optimizer["BackendEstimatorV1 (Wrapper)"].append(None)

In [ ]:
save_data(results_comparison_optimizer, "H2_vqe_estimators_comparison_results_optimizer")

In [ ]:
df_results = pd.DataFrame(results_comparison_optimizer, index=distances)
df_results.to_csv("H2_vqe_estimators_comparison_results_optimizer.csv")

In [ ]:
VQE_comparison_estimators()

In [ ]:
distances = [0.4, 0.74, 1.0, 1.5, 2.0, 2.5]

results = []

for d in tqdm(distances):
    
    results.append(VQE_comparison_estimators(d))

save_dir = os.path.join("data", "pickles")
save_path = os.path.join(save_dir, "h2_qiskit_uccsd_results.pkl")
os.makedirs(save_dir, exist_ok=True)

with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved to {save_path}")


In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])

results = []

for d in tqdm(distances):
    
    results.append(VQE_comparison_estimators_Optimizer(d, method = "ref_sampling", shots=1024))

save_data(results, "h2_qiskit_uccsd_results_sampling.pkl")

In [ ]:
distances = np.concatenate([
    np.linspace(0.3, 0.49, 5), 
    np.linspace(0.5, 1.0, 25), 
    np.linspace(1.1, 2.5, 10)
])

results = []

for d in tqdm(distances):
    
    results.append(VQE_comparison_estimators_Optimizer(d, method = "aer_sampling", shots=1024))

save_data(results, "h2_qiskit_uccsd_results_sampling_dense.pkl")


In [ ]:
distances = np.concatenate([
    np.linspace(0.3, 0.49, 5), 
    np.linspace(0.5, 1.0, 25), 
    np.linspace(1.1, 2.5, 10)
])

results = []

for d in tqdm(distances):
    
    results.append(VQE_comparison_estimators_Optimizer(d, method = "aer_sampling", shots=4096))

save_data(results, "h2_qiskit_uccsd_results_sampling_dense_4096.pkl")


# 3. Optimizer Comparison - Exact & Sampling

In [ ]:
def run_vqe_optimizer_test(dist, method, optimizer, shots=None, seed=42):

    start_time = time.time()
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    if method == "ref_exact":
        estimator = EstimatorV1()
        
    elif method == "ref_sampling":
        estimator = EstimatorV1(options={"shots": shots, "seed": seed})
        
    elif method == "aer_exact":
        estimator = AerEstimator(
            run_options={"shots": None}, 
            approximation=True
        )

    elif method == "aer_sampling":
        estimator = AerEstimator(
            run_options={"shots": shots, "seed": seed}, 
            approximation=False
        )

    elif method == "backend_wrapper":
        sim_backend = AerSimulator()
        estimator = BackendEstimatorV1(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)
        
    else:
        raise ValueError(f"Unknown method: {method}")


    initial_point = [0.0] * ansatz.num_parameters
    
    vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_point)
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time
    
    metrics = {}
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
            
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
            
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

optimizers = {
    "SLSQP":       SLSQP(maxiter=100),
    "COBYLA":      COBYLA(maxiter=100),
    "SPSA":        SPSA(maxiter=100),  
    "Nelder-Mead": NELDER_MEAD(maxiter=100),
    "L-BFGS-B":    L_BFGS_B(maxiter=100),
    "TNC":         TNC(maxiter=100),
    "AQGD":        AQGD(maxiter=100, eta=0.1) 
}

methods = ["ref_exact", "ref_sampling", "aer_exact", "aer_sampling"]

results = {m: {opt: [] for opt in optimizers} for m in methods}

for method in methods:
    print(f"\nMethod: {method}")
    
    for opt_name, opt_obj in tqdm(optimizers.items()):
        
        for d in distances:
        
            res = run_vqe_optimizer_test(
                dist=d, 
                method=method, 
                optimizer=opt_obj, 
                shots=1024
            )
            
            results[method][opt_name].append(res)

save_data(results, "h2_optimiser_comparison_1024.pkl")

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

results_exact = []

for d in distances:

    results_exact.append(VQE_comparison_estimators_Optimizer(dist=d, method="aer_exact"))

print(results_exact)

save_data(results_exact, "h2_optimiser_comparison_1024_exact.pkl")

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

optimizers = {
    "SLSQP":       SLSQP(maxiter=100),
    "COBYLA":      COBYLA(maxiter=100),
    "SPSA":        SPSA(maxiter=100),  
    "Nelder-Mead": NELDER_MEAD(maxiter=100),
    "L-BFGS-B":    L_BFGS_B(maxiter=100),
    "TNC":         TNC(maxiter=100),
    "AQGD":        AQGD(maxiter=100, eta=0.1) 
}

methods = ["ref_exact", "ref_sampling", "aer_exact", "aer_sampling"]

results = {m: {opt: [] for opt in optimizers} for m in methods}

for method in methods:
    print(f"\nMethod: {method}")
    
    for opt_name, opt_obj in tqdm(optimizers.items()):
        
        for d in distances:
        
            res = run_vqe_optimizer_test(
                dist=d, 
                method=method, 
                optimizer=opt_obj, 
                shots=4096
            )
            
            results[method][opt_name].append(res)

save_data(results, "h2_optimiser_comparison_4096.pkl")

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

optimizers = {
    "SLSQP":       SLSQP(),
    "COBYLA":      COBYLA(),
    "SPSA":        SPSA(),  
    "Nelder-Mead": NELDER_MEAD(),
    "L-BFGS-B":    L_BFGS_B(),
    "TNC":         TNC(),
    "AQGD":        AQGD() 
}

methods = ["ref_exact", "ref_sampling", "aer_exact", "aer_sampling"]

results = {m: {opt: [] for opt in optimizers} for m in methods}

for method in methods:
    print(f"\nMethod: {method}")
    
    for opt_name, opt_obj in tqdm(optimizers.items()):
        
        for d in distances:
        
            res = run_vqe_optimizer_test(
                dist=d, 
                method=method, 
                optimizer=opt_obj, 
                shots=4096
            )
            
            results[method][opt_name].append(res)

save_data(results, "h2_optimiser_comparison_4096_nomaxiter.pkl")

In [ ]:
def run_vqe_optimizer_test_V2(dist, method, optimizer, shots=None, seed=42):
    start_time = time.time()
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    if method == "ref_exact":
        estimator = EstimatorV1()
    elif method == "ref_sampling":
        estimator = EstimatorV1(options={"shots": shots, "seed": seed})
    elif method == "aer_exact":
        estimator = AerEstimator(run_options={"shots": None}, approximation=True)
    elif method == "aer_sampling":
        estimator = AerEstimator(run_options={"shots": shots, "seed": seed}, approximation=False)
    elif method == "backend_wrapper":
        sim_backend = AerSimulator()
        estimator = BackendEstimatorV1(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)
    else:
        raise ValueError(f"Unknown method: {method}")

    callback_history = []
    nuc_repulsion = problem.nuclear_repulsion_energy
    
    def store_intermediate_result(eval_count, parameters, mean, std):
        callback_history.append(mean + nuc_repulsion)

    initial_point = [0.0] * ansatz.num_parameters
    
    vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_point, callback=store_intermediate_result)
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time
    
    metrics = {}
    metrics['energy_history'] = callback_history
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
        
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        
        if hasattr(opt_res, 'nit') and opt_res.nit is not None:
            metrics['nit'] = opt_res.nit
        else:
            metrics['nit'] = len(callback_history)
            
        if hasattr(opt_res, 'nfev') and opt_res.nfev is not None:
            metrics['nfev'] = opt_res.nfev
        else:
            metrics['nfev'] = len(callback_history)
            
        if hasattr(opt_res, 'fun') and opt_res.fun is not None:
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)])
distances = np.sort(distances)

optimizers = {
    "SLSQP":       SLSQP(),
    "COBYLA":      COBYLA(),
    "SPSA":        SPSA(),  
    "Nelder-Mead": NELDER_MEAD(),
    "L-BFGS-B":    L_BFGS_B(),
    "TNC":         TNC(),
    "AQGD":        AQGD() 
}

methods = ["ref_exact", "ref_sampling", "aer_exact", "aer_sampling"]

results = {m: {opt: [] for opt in optimizers} for m in methods}

for method in methods:
    print(f"\nMethod: {method}")
    
    for opt_name, opt_obj in tqdm(optimizers.items()):
        
        for d in distances:
        
            res = run_vqe_optimizer_test_V2(
                dist=d, 
                method=method, 
                optimizer=opt_obj, 
                shots=4096
            )
            
            results[method][opt_name].append(res)

save_data(results, "h2_optimiser_comparison_4096_nomaxiter_V2.pkl")

In [ ]:
import time
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_algorithms import VQE
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit.primitives import Estimator as EstimatorV1
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendEstimator as BackendEstimatorV1

def run_vqe_optimizer_with_callback(dist, method, optimizer, shots=None, seed=42):
    start_time = time.time()
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    if method == "ref_exact":
        estimator = EstimatorV1()
    elif method == "ref_sampling":
        estimator = EstimatorV1(options={"shots": shots, "seed": seed})
    elif method == "aer_exact":
        estimator = AerEstimator(run_options={"shots": None}, approximation=True)
    elif method == "aer_sampling":
        estimator = AerEstimator(run_options={"shots": shots, "seed": seed}, approximation=False)
    elif method == "backend_wrapper":
        sim_backend = AerSimulator()
        estimator = BackendEstimatorV1(backend=sim_backend)
        estimator.set_options(shots=shots, seed=seed)
    else:
        raise ValueError(f"Unknown method: {method}")

    energy_history = []
    
    def store_intermediate_result(eval_count, parameters, mean, std):
        energy_history.append(mean)

    initial_point = [0.0] * ansatz.num_parameters
    
    vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_point, callback=store_intermediate_result)
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time
    
    metrics = {}
    metrics['energy_history'] = energy_history
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
        
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        elif hasattr(opt_res, 'nfev'):
            metrics['nit'] = opt_res.nfev
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

energy, elapsed, metrics = run_vqe_optimizer_with_callback(
    dist=0.74, 
    method="aer_sampling", 
    optimizer=SPSA(maxiter=100), 
    shots=4096
)

set_style("production")
fig, ax = plt.subplots(figsize=(10, 6))

history = metrics['energy_history']
ax.plot(range(1, len(history) + 1), history, color="#1f77b4", lw=2, alpha=0.9, label="SPSA")
ax.axhline(-1.1373, color="black", linestyle="--", lw=2, label="Exact FCI (-1.1373 Ha)")

ax.set_title(r"SPSA Optimizer Convergence Tracker ($d = 0.74 \AA$, 4096 Shots)")
ax.set_xlabel("Optimizer Evaluations")
ax.set_ylabel("Energy (Ha)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=True, fancybox=True, edgecolor='black')

plt.show()

# 4. Basis set Comparison - Exact V1

In [ ]:
def VQE_comparison_basis_set(dist, basis, shots=1024, seed=42):
    
    start_time = time.time()

    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
    
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    estimator = AerEstimator(
            run_options={"shots": shots, "seed": seed},
            approximation=True
    )
    
    estimator = EstimatorV1()
    
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time

    metrics = {}
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
            
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
            
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
BASIS_SETS = ["sto-3g", "sto-6g", "6-31g", "6-311g", "cc-pvdz", "cc-pvtz"]

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results = {basis: [] for basis in BASIS_SETS}

for basis in BASIS_SETS:
    print(f"\nTesting Basis: {basis.upper()}")
    for d in tqdm(distances):
        energy = VQE_comparison_basis_set(d, basis)
        results[basis].append(energy)

#save_data(results, "h2_basis_set_comparison_exact.pkl")

In [ ]:
save_data(results, "h2_basis_set_comparison_exact_sto-3g_sto-6g_6-31g_6-311g.pkl")

In [ ]:
BASIS_SETS = ["cc-pvdz", "cc-pvtz"]

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results = {basis: [] for basis in BASIS_SETS}

for basis in BASIS_SETS:
    print(f"\n Basis: {basis.upper()}")
    for d in tqdm(distances):
        energy = VQE_comparison_basis_set(d, basis)
        results[basis].append(energy)

save_data(results, "h2_basis_set_comparison_exact_cc-pvdz_cc-pvtz.pkl")

In [ ]:
BASIS_SETS = ["cc-pvtz"]

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results = {basis: [] for basis in BASIS_SETS}

for basis in BASIS_SETS:
    print(f"\n Basis: {basis.upper()}")
    for d in tqdm(distances):
        energy = VQE_comparison_basis_set(d, basis)
        results[basis].append(energy)

save_data(results, "h2_basis_set_comparison_exact_cc-pvdz_cc-pvtz.pkl")

In [ ]:
def VQE_comparison_basis_set_Aer(dist, basis):
    
    start_time = time.time()

    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
    
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    estimator = AerEstimator(
            run_options={"shots": None, "method": "statevector"},
            approximation=True
    )
    
    
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time

    metrics = {}
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
            
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
            
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
BASIS_SETS = ["sto-3g", "sto-6g", "6-31g", "6-311g"]

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results = {basis: [] for basis in BASIS_SETS}

for basis in BASIS_SETS:
    print(f"\nTesting Basis: {basis.upper()}")
    for d in tqdm(distances):
        energy = VQE_comparison_basis_set_Aer(d, basis)
        results[basis].append(energy)

save_data(results, "h2_basis_set_comparison_exact_Aer.pkl")

# 5. Shot Comparison - Sampling Aer 

In [ ]:
def VQE_shots_analysis(dist, shots, seed=42):
    
    start_time = time.time()

    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    estimator = AerEstimator(
        run_options={"shots": int(shots), "seed": seed}, 
        approximation=False
    )
    
    vqe = VQE(estimator, ansatz, SPSA(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time

    metrics = {}
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

shot_counts = np.arange(100, 5100, 100)

results = {int(shots): [] for shots in shot_counts}

for shots in tqdm(shot_counts, desc="shots"):
    
    for d in distances:
        energy, elapsed, metrics = VQE_shots_analysis(
            dist=d, 
            shots=int(shots)
        )
        
        results[int(shots)].append((energy, elapsed, metrics))

save_data(results, "h2_shots_comparison_fullcurves_5000.pkl")

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

shot_counts = np.arange(100, 7050, 50)

results = {int(shots): [] for shots in shot_counts}

for shots in tqdm(shot_counts, desc="shots"):
    
    for d in distances:
        energy, elapsed, metrics = VQE_shots_analysis(
            dist=d, 
            shots=int(shots)
        )
        
        results[int(shots)].append((energy, elapsed, metrics))

save_data(results, "h2_shots_comparison_fullcurves_7000_50.pkl")

# 6. Noise model Comparison - Sampling Aer and EstimatorV1

In [ ]:
def VQE_noise_comparison(dist, noise_model, shots=1024, seed=42):
    start_time = time.time()
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    estimator = AerEstimator(
        run_options={"shots": shots, "seed": seed},
        backend_options={"noise_model": noise_model} if noise_model else {},
        approximation=False
    )
    
    vqe = VQE(estimator, ansatz, SPSA(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time
    
    metrics = {}
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
        
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], elapsed, metrics

In [ ]:
backend_manila = FakeManilaV2()
backend_jakarta = FakeJakartaV2()
backend_torino = FakeTorino()
backend_kyoto = FakeKyoto()
backend_osaka = FakeOsaka()
backend_brisbane = FakeBrisbane()

noise_models = {
    "Ideal_Shot_Noise": None,
    "Fake_Manila_5Q": NoiseModel.from_backend(backend_manila), 
    "Fake_Jakarta_7Q": NoiseModel.from_backend(backend_jakarta),
    "Fake_Torino_133Q": NoiseModel.from_backend(backend_torino),
    "Fake_Kyoto_127Q": NoiseModel.from_backend(backend_kyoto),
    "Fake_Osaka_127Q": NoiseModel.from_backend(backend_osaka),
    "Fake_Brisbane_127Q": NoiseModel.from_backend(backend_brisbane)
}

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results = {name: [] for name in noise_models.keys()}

for name, noise_model in noise_models.items():
    for d in tqdm(distances, desc=name):
        energy, elapsed, metrics = VQE_noise_comparison(d, noise_model)
        results[name].append((energy, elapsed, metrics))

save_data(results, "h2_noise_comparison.pkl")

# 7. Method Comparison - HF, DFT, FCI, Exact - Statevector Diag, VQE - No sampling.  

In [ ]:
def Run_PySCF_HF(dist, basis='sto-3g'):
    try:
        mol = gto.M(atom=f'H 0 0 0; H 0 0 {dist}', basis=basis, verbose=0)
        mf = scf.RHF(mol)
        return mf.kernel()
    except Exception as e:
        print(f"HF failed at {dist}: {e}")
        return np.nan

def Run_PySCF_DFT(dist, basis='sto-3g'):
    try:
        mol = gto.M(atom=f'H 0 0 0; H 0 0 {dist}', basis=basis, verbose=0)
        mf = dft.RKS(mol)
        mf.xc = 'b3lyp'
        return mf.kernel()
    except Exception as e:
        print(f"DFT failed at {dist}: {e}")
        return np.nan

def Run_PySCF_FCI(dist, basis='sto-3g'):
    try:
        mol = gto.M(atom=f'H 0 0 0; H 0 0 {dist}', basis=basis, verbose=0)
        mf = scf.RHF(mol)
        mf.kernel()
        cisolver = fci.FCI(mf)
        e_fci, _ = cisolver.kernel()
        return e_fci
    except Exception as e:
        print(f"FCI failed at {dist}: {e}")
        return np.nan

def Run_Qiskit_Exact(dist, basis='sto-3g'):
    try:
        driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
        problem = driver.run()
        hamiltonian_op = problem.hamiltonian.second_q_op()
        
        mapper = JordanWignerMapper()
        qubit_op = mapper.map(hamiltonian_op)
        matrix = qubit_op.to_matrix()
        
        eigenvalues = np.linalg.eigvalsh(matrix)
        electronic_energy = eigenvalues[0]
        total_energy = electronic_energy + problem.nuclear_repulsion_energy
        
        return total_energy.real
    except Exception as e:
        print(f"Qiskit Exact failed at {dist}: {e}")
        return np.nan

def Run_Qiskit_VQE(dist, basis='sto-3g'):
    try:
        driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis=basis, charge=0, spin=0)
        problem = driver.run()
        mapper = JordanWignerMapper()
        
        ansatz = UCCSD(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
            initial_state=HartreeFock(
                problem.num_spatial_orbitals,
                problem.num_particles,
                mapper,
            ),
        )
        
        estimator = AerEstimator(
            run_options={"shots": None},
            backend_options={"method": "statevector"},
            approximation=True
        )
        
        vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
        vqe.initial_point = [0.0] * ansatz.num_parameters
        
        calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
        result = calc.solve(problem)
        return result.total_energies[0]
    except Exception as e:
        print(f"Qiskit VQE failed at {dist}: {e}")
        return np.nan

In [ ]:
distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

basis_set = 'sto-3g'

results = {
    "PySCF_HF": [],
    "PySCF_DFT": [],
    "PySCF_FCI": [],
    "Qiskit_Exact": [],
    "Qiskit_VQE": []
}

for d in tqdm(distances):
    results["PySCF_HF"].append(Run_PySCF_HF(d, basis_set))
    results["PySCF_DFT"].append(Run_PySCF_DFT(d, basis_set))
    results["PySCF_FCI"].append(Run_PySCF_FCI(d, basis_set))
    results["Qiskit_Exact"].append(Run_Qiskit_Exact(d, basis_set))
    results["Qiskit_VQE"].append(Run_Qiskit_VQE(d, basis_set))

save_data(results, "h2_method_comparison.pkl")

# 8. SPSA Analysis - Sampling and Exact Aer 

In [ ]:
def VQE_Optimizer_Study(dist, optimizer_cls, optimizer_name, maxiter, mode='aer_exact'):

    start_time = time.time()
    
    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    history = {"eval_count": [], "energy": []}
    
    def callback(nfev, parameters, mean, std):

        history["eval_count"].append(nfev)
        history["energy"].append(mean)
        
    shots = None if mode == 'aer_exact' else 4096
    estimator = AerEstimator(
        run_options={"shots": shots},
        backend_options={"method": "statevector"} if mode == 'aer_exact' else {},
        approximation=(mode == 'aer_exact')
    )
    
    optimizer_instance = optimizer_cls(maxiter=maxiter)
    
    vqe = VQE(
        estimator, 
        ansatz, 
        optimizer_instance, 
        callback=callback
    )
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    
    try:
        result = calc.solve(problem)
        final_energy = result.total_energies[0]
        raw_result = result.raw_result
    except Exception as e:
        print(f"Error with {optimizer_name}: {e}")
        return None, 0, {}

    elapsed = time.time() - start_time
    
    metrics = {
        'cost_function_evals': getattr(raw_result, 'cost_function_evals', len(history['eval_count'])),
        'nit': getattr(raw_result.optimizer_result, 'nit', maxiter),
        'history': history
    }
    
    return final_energy, elapsed, metrics

In [ ]:
OPTIMIZERS = {
    "SLSQP":       SLSQP,
    "COBYLA":      COBYLA,
    "SPSA":        SPSA,  
    "Nelder-Mead": NELDER_MEAD,
    "L-BFGS-B":    L_BFGS_B,
    "TNC":         TNC,
    "AQGD":        AQGD 
}


fixed_distance = 0.74
maxiters_to_test = [50, 100, 200, 500]
modes = ['aer_exact', 'aer_sampling']

results = {mode: {opt: {} for opt in OPTIMIZERS.keys()} for mode in modes}

for mode in modes:
    for opt_name, opt_cls in tqdm(OPTIMIZERS.items()):
        for m_iter in tqdm(maxiters_to_test):
            energy, elapsed, metrics = VQE_Optimizer_Study(
                fixed_distance, 
                opt_cls, 
                opt_name, 
                m_iter, 
                mode=mode
            )
            
            results[mode][opt_name][f"maxiter_{m_iter}"] = (energy, elapsed, metrics)

save_data(results, "h2_optimizer_comparison_V2.pkl")

# 9. Shot Comparison V2 - Sampling Aer 

In [ ]:
def VQE_shots_optimizer_test(dist, shots, seed=42):
    start_time = time.time()

    driver = PySCFDriver(atom=f"H 0 0 0; H 0 0 {dist}", basis='sto-3g', charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            problem.num_spatial_orbitals,
            problem.num_particles,
            mapper,
        ),
    )
    
    estimator = AerEstimator(run_options={"shots": int(shots), "seed": seed}, approximation=False)
    
    optimizer = SPSA(maxiter=500)
    vqe = VQE(estimator, ansatz, optimizer)
    
    np.random.seed(seed)
    vqe.initial_point = np.random.uniform(-0.01, 0.01, ansatz.num_parameters)
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(problem)
    
    elapsed = time.time() - start_time

    metrics = {}
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        metrics['nit'] = getattr(opt_res, 'nit', np.nan)
        metrics['nfev'] = getattr(opt_res, 'nfev', np.nan)
            
    return result.total_energies[0], elapsed, metrics

distances = np.sort(np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)]))

shot_counts_A = np.arange(100, 7050, 50) 
results_A = {int(shots): [] for shots in shot_counts_A}

for shots in tqdm(shot_counts_A):
    for d in distances:
        energy, elapsed, metrics = VQE_shots_optimizer_test(d, int(shots))
        results_A[int(shots)].append((energy, elapsed, metrics))

In [ ]:
save_data(results_A, "h2_shots_comparison_fullcurves_7000_50_SPSA500.pkl")